In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

#import personnal tools
import sys
sys.path.append('../tools/')
from info import *
from tools_generic import *
from events import *

# Load files

In [ ]:
site_list=["d17","d47","d85","dmc"]
start_date = '20241201'
end_date = '20260630'

In [ ]:
data = {}
daily_data = {}

In [ ]:
data = create_data(site_list, sensors, start_date, end_date)

# Add the early WIND observations stored in separate WIND_BEGINNING files.
# WARNING: Existing *_wind values take priority; beginning-file values only fill gaps.
data = add_wind_beginning_data(
    data,
    sites=["d17", "d47", "d85", "dmc"],
    data_folder_path="../../data",
    start_date="20241201",
    end_date="20250313",
)

data

In [ ]:
# create daily
daily_data = create_daily_data(data)
# daily_data

# Stats

In [ ]:
variable = "FluxMean1_flowcapt"
compute_variable_stats(data, variable)


In [ ]:
var = "FluxMean1_flowcapt"
plot_binned_distribution(data, var, bin_number=30, min_value=0, max_value=200)


In [ ]:
var = "snowflux_spc"
plot_binned_distribution(data, var, bin_number=30, min_value=0, max_value=200)


# Plotting

In [ ]:
data["d17"]

In [ ]:
variables = ["wspd1_wind",'wdir_wind']

plot_per_var_multiple_sites(
    sensor_datasets=data,
    variables=variables,
    sites=["d17", "d47", 'd85','dmc'],
    figsize=(15, 5),
    # ymin=[-0.1],
    # ymax=[3],
)


In [ ]:
variables = ["FluxMean1_flowcapt", "FluxMean2_flowcapt", "snowflux_spc", "Hagl_flowcapt"]
variables = ["wspd1_wind", "wdir_wind"]

plot_monthly_availability_table(
    sensor_datasets=data,
    variables=variables,
    sites=site_list,
    figsize=(16, 2 * len(variables)),
)


# Events

In [ ]:
detector = EventDetector(
    threshold=1.0,
    min_timesteps=12,
    buffer_timesteps=0,
)

sampled_data = create_resampled_data(data, "30min")
additional_variables = [
    "FluxMean1_flowcapt",
    "snowflux_spc",
    "wspd1_wind",
    "wspd2_wind",
    "wdir_wind",
    "Hagl_flowcapt",
    "T1_surf",
    "RH1_surf",
]
collection = detector.detect_events(
    sampled_data,
    variable="FluxMean2_flowcapt",
    additional_variables=additional_variables,
)
collection_d17 = collection.get_site("d17")
collection_d47 = collection.get_site("d47")
non_collection = detector.detect_non_events(
    sampled_data,
    variable="FluxMean2_flowcapt",
    additional_variables=additional_variables,
)
non_collection_d17 = non_collection.get_site("d17")
non_collection_d47 = non_collection.get_site("d47")


In [ ]:
collection_d17.to_catalog()

In [ ]:
varlist = ['FluxMean2_flowcapt','FluxMean1_flowcapt','snowflux_flowcapt']
# varlist = ['FluxMean2', 'wspd1','wdir']
# varlist = ['Hagl','T1','RH1']
plot_events_vs_nonevents_chronological(
    collection_d17,
    non_collection_d17,
    variables=varlist,
    figsize=(20,10)
)
plot_events_vs_nonevents_chronological(
    collection_d47,
    non_collection_d47,
    variables=varlist,
    figsize=(20,10)
)

In [ ]:
plot_starting_month_distribution(collection_d17)
plot_starting_month_distribution(collection_d47)

In [ ]:
plot_diurnal_start_distribution(collection_d17)
plot_diurnal_start_distribution(collection_d47)